# Generating a Full CivBench Test Suite

This notebook builds a multi-program, multi-state CivBench evaluation suite:

- SNAP across VA (BBCE), CA (BBCE), TX (strict asset test)
- WIC across VA and national thresholds
- ~200 total cases
- Exported as YAML, JSONL, and CSV

This is the workflow for generating a suite ready to submit to the CivBench benchmark.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth import Pipeline, BatchPipeline, list_presets
from collections import Counter
import json

os.makedirs('./suite_output', exist_ok=True)
print('Ready.')

## 1. Available Presets

In [ ]:
list_presets()

## 2. Generate SNAP Cases — Three States

VA and CA both have BBCE (no asset test). TX uses the strict federal asset test.
This creates meaningful variation in the case pool.

In [ ]:
snap_batch = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
snap_cases = snap_batch.generate(n_per_pipeline=40, seed=42)

print(f'SNAP cases generated: {len(snap_cases)}')
print()
by_state = Counter(c.jurisdiction for c in snap_cases)
print('By jurisdiction:')
for j, n in sorted(by_state.items()):
    elig = sum(1 for c in snap_cases if c.jurisdiction == j and c.expected_outcome == 'eligible')
    inelig = n - elig
    print(f'  {j:<10} {n:>3} cases  ({elig} eligible, {inelig} ineligible)')
print()
by_diff = Counter(c.difficulty.value for c in snap_cases)
print('Difficulty:', dict(by_diff))


## 3. Generate WIC Cases

In [ ]:
wic_batch = BatchPipeline.from_presets(['wic.national'])
wic_cases = wic_batch.generate(n_per_pipeline=30, seed=42)

print(f'WIC cases generated: {len(wic_cases)}')
by_state = Counter(c.jurisdiction for c in wic_cases)
for j, n in sorted(by_state.items()):
    print(f'  {j}: {n} cases')


## 4. Combine and Summarize the Full Suite

In [ ]:
all_cases = snap_cases + wic_cases

print(f'Total cases: {len(all_cases)}')
print()

by_program = Counter(c.program for c in all_cases)
by_difficulty = Counter(c.difficulty.value for c in all_cases)
by_outcome = Counter(c.expected_outcome for c in all_cases)

print('By program:')
for p, n in sorted(by_program.items()):
    print(f'  {p:<10} {n:>4} cases')

print()
print('By difficulty:')
for d in ['easy', 'medium', 'hard', 'adversarial']:
    n = by_difficulty.get(d, 0)
    pct = n / len(all_cases) * 100
    bar = '█' * int(pct / 2)
    print(f'  {d:<12} {n:>4}  ({pct:>5.1f}%)  {bar}')

print()
print('By outcome:')
for o, n in by_outcome.items():
    print(f'  {o:<12} {n:>4} ({n/len(all_cases)*100:.0f}%)')


## 5. Verify All Case IDs Are Unique

In [ ]:
ids = [c.case_id for c in all_cases]
unique_ids = set(ids)
duplicate_ids = [id for id in ids if ids.count(id) > 1]

print(f'Total cases:   {len(all_cases)}')
print(f'Unique IDs:    {len(unique_ids)}')
print(f'Duplicates:    {len(duplicate_ids)}')

if duplicate_ids:
    print('\nDuplicate IDs:')
    for d in set(duplicate_ids):
        print(f'  {d}')
else:
    print('\n✓ All case IDs are unique')


## 6. Run CivBench Compatibility Checks

In [ ]:
errors = []
for case in all_cases:
    case_errors = case.validate()
    if case_errors:
        errors.append((case.case_id, case_errors))

if errors:
    print(f'Validation errors in {len(errors)} cases:')
    for cid, errs in errors[:5]:
        print(f'  {cid}: {errs}')
else:
    print(f'✓ All {len(all_cases)} cases pass validation')


## 7. Export the Suite — All Formats

In [ ]:
# Use a pipeline to save (BatchPipeline.save takes the full case list)
snap_pipeline = Pipeline.from_preset('snap.va')

# YAML — one file per case (the canonical format)
snap_pipeline.save(all_cases, './suite_output/yaml/', formats=['yaml'])

# JSONL — convenient for fine-tuning datasets
snap_pipeline.save(all_cases, './suite_output/civbench_suite.jsonl', formats=['jsonl'])

# CSV — for human review / spreadsheet analysis
snap_pipeline.save(all_cases, './suite_output/civbench_suite.csv', formats=['csv'])

print('Output files:')
for root, dirs, files in os.walk('./suite_output'):
    level = root.replace('./suite_output', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files)[:10]:  # cap at 10 per dir
        print(f'{indent}  {f}')
    if len(files) > 10:
        print(f'{indent}  ... and {len(files)-10} more')


## 8. Preview JSONL — Structure for Fine-Tuning

In [ ]:
with open('./suite_output/civbench_suite.jsonl') as f:
    first = json.loads(f.readline())

print('Top-level keys:', list(first.keys()))
print()
print('case_id:', first['case_id'])
print('program:    ', first['metadata']['program'])
print('difficulty: ', first['metadata']['difficulty'])
print('outcome:    ', first['metadata']['expected_outcome'])
print()
print('scenario (from user message):')
print(' ', first['messages'][1]['content'][:120], '...')
print()
print('message roles:', [m['role'] for m in first['messages']])
